In [ ]:
# --- Standard Library Imports ---
import os
import sys
import shutil
import warnings
import logging
import time
from pathlib import Path
from typing import Optional, Callable, Dict, Any, List
import pandas as pd


# --- Suppress TensorFlow and Addon Warnings for Cleaner Console ---
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow_addons')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# --- Local Application Imports ---
from common.config_wells import DATA_SOURCES, filter_data_sources
from forecast_pipeline.config import ARCHITECTURE_YAML_PATH, ARCH
from forecast_pipeline.config import (
    DEFAULT_DATASET,
    MAX_WORKERS,
    DefaultExperimentParams,
    PROFILES_DIR,
    ARCHITECTURE_YAML_PATH,
    HPO_STUDIES_DIR,
    EXPERIMENTS_OUTPUT_DIR
)
from forecast_pipeline.jobs import (
    generate_jobs,
    run_single_job,
    create_filter_configurations,
    select_data_sources
)
from forecast_pipeline.io_utils import (
    configure_logging,
    save_experiment_to_excel,
    generate_experiment_name
)
from forecast_pipeline.runner import (
    execute_jobs,
    execute_jobs_robust
)
from forecast_pipeline.metrics import (
    collate_robust_results,
    collate_metrics,
    clean_and_structure_results
)
from hpo.optuna_utils import (
    generate_trials_from_study,
    report_results_to_study
)
from hpo.search_space import (
    define_fast_search_space,
    define_seq2context_space,
    define_seq2pin_family_space
)
from functools import partial
from hpo.analysis_utils import add_weighted_score
import optuna
import optuna.visualization as vis

# --- Pandas Display Settings ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

# --- Configure Logging ---
configure_logging()

In [ ]:
# ==============================================================================
#                         HPO CAMPAIGN CONTROL PANEL
# ==============================================================================
DATA_SOURCES = filter_data_sources(DATA_SOURCES, dataset_name="VOLVE", well_name="15/9-F-14")
CAMPAIGN_MODE = 'FULL_SWEEP'  # Options: 'FAST_DEBUG', 'FULL_SWEEP'
ensemble_size = 1

if CAMPAIGN_MODE == 'FAST_DEBUG':
    STUDY_NAME = "F12_FAST"
    SEARCH_SPACE_FUNC = define_fast_search_space
    TRIALS_PER_CYCLE_SCHEDULE = [100, 50, 30]
else:  # FULL_SWEEP
    STUDY_NAME = "F14_TREND_Deterministic"
    TRIALS_PER_CYCLE_SCHEDULE = [200, 100, 50, 50]
    TRIALS_PER_CYCLE_SCHEDULE = [5, 3, 2]
    if ARCH == "Seq2Context":
        SEARCH_SPACE_FUNC = define_seq2context_space
    else: # For Seq2Trend, Seq2PIN, etc.
        SEARCH_SPACE_FUNC = define_seq2pin_family_space
    
SEARCH_SPACE_FUNC = partial(SEARCH_SPACE_FUNC, architecture_yaml_path=ARCHITECTURE_YAML_PATH)
METRIC_WEIGHTS = {
    'val_smape_cum': 2.0,
    'val_smape_agg': 2.0
}
LOWER_IS_BETTER = {
    'val_smape_cum': True,
    'val_smape_agg': True
}
from forecast_pipeline.config import ARCH
METRIC_TO_OPTIMIZE = 'weighted_score'
FIXED_PARAMS = {"seed": 42,  "architecture_name": ARCH}

print("✅ Setup complete. Ready to orchestrate HPO campaign.")
print(f"Running in {CAMPAIGN_MODE} mode.")

# ==============================================================================
#                             Pipeline Functions
# ==============================================================================

def run_robust_pipeline(profile_path: str, ensemble_size: int) -> Optional[str]:
    """Robust, profile-driven workflow for a single batch of jobs."""
    logging.info(f"--- Starting pipeline in ROBUST mode for profile: {profile_path} ---")
    sources = select_data_sources(DATA_SOURCES, DEFAULT_DATASET)
    default_params = {**vars(DefaultExperimentParams()), "ensemble_models": ensemble_size}

    jobs = generate_jobs(sources, default_params, profile_path=profile_path)
    if not jobs:
        logging.warning("No jobs generated from profile. Exiting.")
        return None

    run_name = Path(profile_path).stem
    run_output_dir = execute_jobs_robust(jobs, "experiments", run_name, MAX_WORKERS)

    logging.info(f"Run complete. Collating results from: {run_output_dir}")
    leaderboard_df = collate_robust_results(run_output_dir)

    if not leaderboard_df.empty:
        leaderboard_path = Path(run_output_dir) / "leaderboard.csv"
        leaderboard_df.to_csv(leaderboard_path, index=False)
        logging.info(f"Leaderboard saved to: {leaderboard_path}")
        print("\n--- Top 5 Results (by val_smape_cum) ---")
        try:
            print(leaderboard_df.sort_values(by="val_smape_cum", ascending=True).head(5))
        except KeyError:
            logging.warning("'val_smape_cum' not in leaderboard. Displaying first 5 rows.")
            print(leaderboard_df.head(5))
    else:
        logging.warning("No successful jobs found to create a leaderboard.")

    return run_output_dir

import time
def run_hpo_campaign(
    study_name: str,
    search_space_func: Callable[[optuna.trial.Trial], Dict[str, Any]],
    pipeline_runner_func: Callable[[str, int], Optional[str]],
    trials_per_cycle_schedule: List[int],
    metric_weights: Dict[str, float],
    lower_is_better: Dict[str, bool],
    metric_to_optimize: str,
    fixed_params: Dict[str, Any],
    ensemble_size: int = 1
) -> pd.DataFrame:
    """Run a full, automated HPO campaign in cycles with result feedback."""
    all_cycle_leaderboards = []
    total_cycles = len(trials_per_cycle_schedule)

    start_time = time.time()  # <-- INÍCIO DA MEDIÇÃO

    for i, num_trials in enumerate(trials_per_cycle_schedule):
        cycle_num = i + 1
        logging.info(f"\n{'='*25} HPO CYCLE {cycle_num}/{total_cycles} {'='*25}")

        profile_path = PROFILES_DIR / f"{study_name}_cycle_{cycle_num}.csv"
        logging.info(f"Generating {num_trials} new trials...")
        generate_trials_from_study(
            study_name=study_name,
            n_trials=num_trials,
            output_file=profile_path,
            search_space_func=search_space_func,
            **fixed_params
        )

        logging.info(f"Executing pipeline for profile: {profile_path}")
        run_output_dir = pipeline_runner_func(profile_path=str(profile_path), ensemble_size=ensemble_size)
        if not run_output_dir:
            logging.warning(f"Pipeline run for cycle {cycle_num} failed. Skipping.")
            continue

        logging.info("Collating results and reporting back to Optuna...")
        cycle_leaderboard_df = collate_robust_results(run_output_dir)
        if cycle_leaderboard_df.empty:
            logging.warning("No successful jobs in this cycle to report.")
            continue

        leaderboard_path = Path(run_output_dir) / "leaderboard.csv"
        cycle_leaderboard_df.to_csv(leaderboard_path, index=False)

        leaderboard_with_score_df = add_weighted_score(
            cycle_leaderboard_df, metric_weights, lower_is_better
        )
        all_cycle_leaderboards.append(leaderboard_with_score_df)

        report_results_to_study(
            study_name=study_name,
            leaderboard_df=leaderboard_with_score_df,
            metric_to_optimize=metric_to_optimize
        )
        logging.info(f"--- End of Cycle {cycle_num} ---")

    end_time = time.time()  # <--
    elapsed_time = end_time - start_time
    print(f"Total HPO campaign runtime: {elapsed_time:.2f} seconds")

    logging.info("\n🎉 HPO Campaign Complete!")

    if all_cycle_leaderboards:
        return pd.concat(all_cycle_leaderboards, ignore_index=True)
    return pd.DataFrame()

# ==============================================================================
#                           Clean Up Old Data
# ==============================================================================

def cleanup_old_study_data(study_name: str):
    """Remove old study DB and result directories to ensure a fresh run."""
    db_path = HPO_STUDIES_DIR / f"{study_name}.db"
    if db_path.exists():
        print(f"--- Removing old study database for a fresh start: {db_path} ---")
        db_path.unlink()

    print(f"--- Searching for and removing old result directories for study: '{study_name}' ---")
    run_name_prefix = study_name
    for old_run_dir in EXPERIMENTS_OUTPUT_DIR.glob(f"*{run_name_prefix}*"):
        if old_run_dir.is_dir():
            print(f"Removing old experiment directory: {old_run_dir}")
            shutil.rmtree(old_run_dir)

# ==============================================================================
#                              Run Campaign
# ==============================================================================

def main():
    cleanup_old_study_data(STUDY_NAME)

    master_leaderboard = run_hpo_campaign(
        study_name=STUDY_NAME,
        search_space_func=SEARCH_SPACE_FUNC,
        pipeline_runner_func=run_robust_pipeline,
        trials_per_cycle_schedule=TRIALS_PER_CYCLE_SCHEDULE,
        metric_weights=METRIC_WEIGHTS,
        lower_is_better=LOWER_IS_BETTER,
        metric_to_optimize=METRIC_TO_OPTIMIZE,
        fixed_params=FIXED_PARAMS,
        ensemble_size=ensemble_size
    )

    if not master_leaderboard.empty:
        print("\n--- Master Leaderboard (Top 10 Overall by Weighted Score) ---")
        display(master_leaderboard.sort_values(by=METRIC_TO_OPTIMIZE).head(10))

    return master_leaderboard

# --- Run Script ---
if __name__ == "__main__":
    master_leaderboard = main()

In [ ]:
# --- Master Leaderboard Analysis & Visualization Module ---

import optuna
from pathlib import Path

def get_study(study_name: str, studies_dir: Path) -> optuna.Study:
    """Load an Optuna study from the given path."""
    storage_path = f"sqlite:///{studies_dir / (study_name + '.db')}"
    print(f"Loading study from: {storage_path}")
    return optuna.load_study(study_name=study_name, storage=storage_path)

def get_hyperparameter_columns(study: optuna.Study) -> list:
    """Extract hyperparameter columns from the first completed trial."""
    for trial in study.trials:
        if trial.state == optuna.trial.TrialState.COMPLETE:
            return list(trial.params.keys())
    return []

def get_unique_best_configs(df, metric: str, hyper_cols: list, n=10):
    """Return the top-N unique best hyperparameter configurations."""
    sorted_df = df.sort_values(by=metric, ascending=True)
    unique_df = sorted_df.drop_duplicates(subset=hyper_cols, keep='first')
    return unique_df.head(n)

def save_leaderboard(df, output_dir: Path, study_name: str):
    """Save the leaderboard DataFrame as CSV."""
    save_dir = output_dir / f"master_leaderboard_{study_name}"
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / "leaderboard.csv"
    df.to_csv(save_path, index=False)
    print(f"Leaderboard saved to: {save_path}")

def display_leaderboard(df, display_cols: list):
    """Display leaderboard with given columns if they exist."""
    cols = [col for col in display_cols if col in df.columns]
    display(df[cols])

def analyze_study(study, metric: str):
    """Analyze and visualize completed Optuna trials."""
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f"\nStudy '{study.study_name}': {len(study.trials)} trials, {len(completed)} completed.")

    if not completed:
        print("No completed trials in the study yet. Please run the pipeline and report results first.")
        return

    # Show the best trial summary
    best = study.best_trial
    print(f"Best trial ({metric}): {best.value:.4f}\nParams:")
    for k, v in best.params.items():
        print(f"  {k}: {v}")

    # Visualizations (assumes vis module available)
    print("\n--- Optimization History ---")
    display(vis.plot_optimization_history(study, target_name=metric))

    print("\n--- Hyperparameter Importances ---")
    from optuna.importance import MeanDecreaseImpurityImportanceEvaluator
    display(vis.plot_param_importances(study, evaluator=MeanDecreaseImpurityImportanceEvaluator()))


def load_leaderboard_from_disk(study_name, output_dir):
    """Tenta carregar o leaderboard do disco a partir dos nomes convencionados."""
    leaderboard_path = output_dir / f"master_leaderboard_{study_name}" / "leaderboard.csv"
    if not leaderboard_path.exists():
        print(f"Leaderboard não encontrado em: {leaderboard_path}")
        return None
    df = pd.read_csv(leaderboard_path)
    print(f"Leaderboard carregado de: {leaderboard_path}")
    return df

def robust_leaderboard_analysis(
    master_leaderboard=None,    # pode ser None!
    study_name=None,
    studies_dir=None,
    output_dir=None,
    metric_to_optimize=None,
    display_cols=None
):
    """
    Função robusta: se master_leaderboard não for passado,
    recarrega do disco usando os nomes/paths do pipeline.
    """
    # Proteção: se não tem master_leaderboard, tenta carregar do disco
    if master_leaderboard is None:
        if not all([study_name, output_dir]):
            raise ValueError("Para recarregar do disco, forneça study_name e output_dir.")
        master_leaderboard = load_leaderboard_from_disk(study_name, output_dir)
        if master_leaderboard is None or master_leaderboard.empty:
            print("Master leaderboard está vazio ou não encontrado. Nada para analisar.")
            return
    
    # Carrega o study
    study = get_study(study_name, studies_dir)
    hyper_cols = get_hyperparameter_columns(study)

    if hyper_cols:
        print(f"\n--- Top 10 UNIQUE Configurations (by '{metric_to_optimize}') ---")
        unique_best = get_unique_best_configs(master_leaderboard, metric_to_optimize, hyper_cols)
        display_leaderboard(unique_best, display_cols)
    else:
        print("Could not determine hyperparameter columns from study. Showing raw top 10.")
        display_leaderboard(master_leaderboard.sort_values(by=metric_to_optimize).head(10), display_cols)

    # Final analysis and visualization
    print("\n--- HPO Visualizations ---")
    analyze_study(study, metric_to_optimize)

    # Salva o leaderboard atualizado (opcional, só se veio da RAM e não do disco)
    save_leaderboard(master_leaderboard, output_dir, study_name)

DISPLAY_COLS = [
    'architecture_profile', 
    'batch_size', 
    'data_sample', 
    'epochs', 
    'lag_window',
    'learning_rate', 
    'physics_strategy',
    'weighted_score', 
    'val_smape_cum', 
    'val_smape_agg', 
]

# EXEMPLO DE USO 1: (logo após pipeline, sem precisar recarregar)
robust_leaderboard_analysis(
    master_leaderboard=master_leaderboard,   # já na RAM
    study_name=STUDY_NAME,
    studies_dir=HPO_STUDIES_DIR,
    output_dir=EXPERIMENTS_OUTPUT_DIR,
    metric_to_optimize=METRIC_TO_OPTIMIZE,
    display_cols=DISPLAY_COLS
)

In [ ]:
# EXEMPLO DE USO 2: (após queda do kernel, sem master_leaderboard em RAM)
robust_leaderboard_analysis(
    master_leaderboard=None,  # vai ser recarregado do disco!
    study_name=STUDY_NAME,
    studies_dir=HPO_STUDIES_DIR,
    output_dir=EXPERIMENTS_OUTPUT_DIR,
    metric_to_optimize=METRIC_TO_OPTIMIZE,
    display_cols=DISPLAY_COLS
)